In [ ]:
# ============================================================
# VALIDATION A — BRANCH A
# D1 — CEDEFOP Labour Skills Shortage Dataset
# ============================================================
#
# Methodology stage covered:
# Stage 4 — Post-processing and Validation
#
# Compares the preserved Branch A extraction against the
# fixed Stage 1 document-grounded reference dataset.
# ============================================================

import json
import hashlib
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import files

In [ ]:
# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

DOCUMENT_ID = "D1"
BRANCH_ID = "A"

EXPECTED_FIELDS = [
    "Geographic Area",
    "Main Occupation Group",
    "Occupation Group (2 digit)",
    "Labour Shortage Index",
    "LSI (Comp.)",
    "LSI1",
    "LSI2",
    "LSI3"
]

MATCH_KEY_FIELDS = [
    "Geographic Area",
    "Occupation Group (2 digit)"
]

NUMERIC_FIELDS = [
    "Labour Shortage Index",
    "LSI1",
    "LSI2",
    "LSI3"
]

TEXT_FIELDS = [
    "Geographic Area",
    "Main Occupation Group",
    "Occupation Group (2 digit)"
]

PRIMARY_CORRECTNESS_FIELDS = [
    field
    for field in EXPECTED_FIELDS
    if field not in MATCH_KEY_FIELDS
]

PRIMARY_MATCH_COLUMNS = [
    f"{field}_match"
    for field in PRIMARY_CORRECTNESS_FIELDS
]

OUTPUT_DIR = Path("outputs_D1_validation_branch_A")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Validation configured.")


In [ ]:
# ------------------------------------------------------------
# 2. Upload validation inputs
# ------------------------------------------------------------
# Upload:
#   1) D1_branch_A_parsed_extraction.json
#   2) D1_branch_A_technical_diagnostics.json
#   3) D1_reference_values.csv

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [f for f in uploaded_files if f.lower().endswith(".csv")]
json_files = [f for f in uploaded_files if f.lower().endswith(".json")]

if len(csv_files) != 1:
    raise ValueError("Upload exactly one Stage 1 reference-values CSV.")

if len(json_files) != 2:
    raise ValueError(
        "Upload exactly two JSON files: the parsed extraction and the Branch A technical diagnostics."
    )

REFERENCE_FILE = csv_files[0]

parsed_extraction_file = None
technical_diagnostics_file = None

for file_name in json_files:
    with open(file_name, "r", encoding="utf-8") as f:
        obj = json.load(f)

    if isinstance(obj, dict) and isinstance(obj.get("records"), list):
        parsed_extraction_file = file_name

    if (
        isinstance(obj, dict)
        and "json_valid" in obj
        and "top_level_checks" in obj
        and "record_structure_issues" in obj
    ):
        technical_diagnostics_file = file_name

if parsed_extraction_file is None:
    raise ValueError("Could not identify the Branch A parsed extraction JSON.")

if technical_diagnostics_file is None:
    raise ValueError("Could not identify the Branch A technical-diagnostics JSON.")

print("Reference values:", REFERENCE_FILE)
print("Parsed extraction:", parsed_extraction_file)
print("Technical diagnostics:", technical_diagnostics_file)

In [ ]:
# ------------------------------------------------------------
# 3. Input loading and provenance
# ------------------------------------------------------------

with open(parsed_extraction_file, "r", encoding="utf-8") as f:
    extraction_json = json.load(f)

with open(technical_diagnostics_file, "r", encoding="utf-8") as f:
    technical_diagnostics = json.load(f)

df_ref = pd.read_csv(REFERENCE_FILE)
df_ext_raw = pd.DataFrame(extraction_json["records"])

if extraction_json.get("document_id") != DOCUMENT_ID:
    raise ValueError(
        f"Unexpected extraction document_id: {extraction_json.get('document_id')}"
    )

if extraction_json.get("branch") != BRANCH_ID:
    raise ValueError(
        f"Unexpected extraction branch: {extraction_json.get('branch')}"
    )

if technical_diagnostics.get("document_id") != DOCUMENT_ID:
    raise ValueError(
        "Technical-diagnostics document_id does not match D1."
    )

if technical_diagnostics.get("branch") != BRANCH_ID:
    raise ValueError(
        "Technical-diagnostics branch does not match Branch A."
    )

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

input_provenance = {
    "reference_file": REFERENCE_FILE,
    "reference_sha256": sha256_file(REFERENCE_FILE),
    "parsed_extraction_file": parsed_extraction_file,
    "parsed_extraction_sha256": sha256_file(parsed_extraction_file),
    "technical_diagnostics_file": technical_diagnostics_file,
    "technical_diagnostics_sha256": sha256_file(technical_diagnostics_file)
}

print("Reference shape:", df_ref.shape)
print("Extraction shape:", df_ext_raw.shape)


In [ ]:
# ------------------------------------------------------------
# 4. Structural validity
# ------------------------------------------------------------

top = technical_diagnostics.get("top_level_checks", {})

required_top_level_checks = [
    "output_is_json_object",
    "document_id_present",
    "document_id_correct",
    "branch_present",
    "branch_correct",
    "records_present",
    "records_is_list"
]

top_level_valid = all(bool(top.get(check, False)) for check in required_top_level_checks)

structural_validity = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

schema_diagnostics = {
    "json_valid": bool(
        technical_diagnostics.get("json_valid", False)
    ),
    "structurally_evaluable":
        structural_validity,
    "records_with_structure_issues": int(
        technical_diagnostics.get(
            "records_with_structure_issues",
            0
        )
    ),
    "numeric_field_type_issues": int(
        technical_diagnostics.get(
            "numeric_field_type_issues",
            0
        )
    )
}

print(json.dumps(schema_diagnostics, indent=2))


In [ ]:
# ------------------------------------------------------------
# 5. Validation-field verification
# ------------------------------------------------------------

missing_reference_fields = [
    field for field in EXPECTED_FIELDS
    if field not in df_ref.columns
]

if missing_reference_fields:
    raise ValueError(
        f"Stage 1 reference dataset is missing fields: {missing_reference_fields}"
    )

missing_extraction_columns = [
    field for field in EXPECTED_FIELDS
    if field not in df_ext_raw.columns
]

df_ext = df_ext_raw.copy()

for field in missing_extraction_columns:
    df_ext[field] = np.nan

df_ref = df_ref[EXPECTED_FIELDS].copy()
df_ext = df_ext[EXPECTED_FIELDS].copy()

print("Missing extraction columns:", missing_extraction_columns)


In [ ]:
# ------------------------------------------------------------
# 6. Comparison normalisation
# ------------------------------------------------------------

def normalize_text(value):
    if pd.isna(value):
        return ""

    value = str(value).strip().lower()
    value = (
        value
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )
    return " ".join(value.split())


def normalize_number(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    value = str(value).strip()
    if value == "":
        return np.nan

    if "," in value and "." not in value:
        value = value.replace(",", ".")

    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def numbers_match(reference_value, extracted_value, decimals=None, tolerance=None):
    ref_num = normalize_number(reference_value)
    ext_num = normalize_number(extracted_value)

    if pd.isna(ref_num) and pd.isna(ext_num):
        return True

    if pd.isna(ref_num) or pd.isna(ext_num):
        return False

    if decimals is not None:
        return round(ref_num, decimals) == round(ext_num, decimals)

    return abs(ref_num - ext_num) <= tolerance

def normalize_lsi_comp(value):
    if pd.isna(value):
        return ""

    value = str(value).strip()
    value = (
        value
        .replace(" ", "")
        .replace("–", "-")
        .replace("—", "-")
    )
    return value

NUMERIC_RULES = {
    "Labour Shortage Index": {"decimals": 2},
    "LSI1": {"tolerance": 0.0},
    "LSI2": {"tolerance": 0.0},
    "LSI3": {"tolerance": 0.0}
}


In [ ]:
# ------------------------------------------------------------
# 7. Record identity construction
# ------------------------------------------------------------

def create_key(dataframe):
    return (
        dataframe["Geographic Area"].apply(normalize_text)
        + " | "
        + dataframe["Occupation Group (2 digit)"].apply(normalize_text)
    )

df_ref["match_key"] = create_key(df_ref)
df_ext["match_key"] = create_key(df_ext)

reference_duplicate_count = int(df_ref["match_key"].duplicated(keep=False).sum())

if reference_duplicate_count > 0:
    duplicate_ref = df_ref[df_ref["match_key"].duplicated(keep=False)]
    raise ValueError(
        "The fixed Stage 1 reference dataset contains duplicate observation keys; "
        "the selected D1 matching key is therefore not unique."
    )

extraction_duplicate_mask = df_ext["match_key"].duplicated(keep="first")
duplicate_extraction_records = df_ext[extraction_duplicate_mask].copy()
df_ext_unique = df_ext[~extraction_duplicate_mask].copy()

print("Reference duplicate observations:", reference_duplicate_count)
print("Additional extracted duplicate observations:", len(duplicate_extraction_records))


In [ ]:
# ------------------------------------------------------------
# 8. One-to-one record alignment
# ------------------------------------------------------------

df_validation = df_ref.merge(
    df_ext_unique,
    on="match_key",
    how="outer",
    suffixes=("_ref", "_ext"),
    indicator=True,
    validate="one_to_one"
)

print(df_validation["_merge"].value_counts(dropna=False))

In [ ]:
# ------------------------------------------------------------
# 9. Field-level comparison
# ------------------------------------------------------------

matched_mask = df_validation["_merge"] == "both"

for field in NUMERIC_FIELDS:
    col = f"{field}_match"
    rule = NUMERIC_RULES[field]

    df_validation[col] = False
    df_validation.loc[matched_mask, col] = df_validation.loc[matched_mask].apply(
        lambda row: numbers_match(
            row[f"{field}_ref"],
            row[f"{field}_ext"],
            **rule
        ),
        axis=1
    )

df_validation["LSI (Comp.)_match"] = False
df_validation.loc[matched_mask, "LSI (Comp.)_match"] = df_validation.loc[
    matched_mask
].apply(
    lambda row:
        normalize_lsi_comp(row["LSI (Comp.)_ref"])
        == normalize_lsi_comp(row["LSI (Comp.)_ext"]),
    axis=1
)

for field in TEXT_FIELDS:
    col = f"{field}_match"
    df_validation[col] = False
    df_validation.loc[matched_mask, col] = df_validation.loc[matched_mask].apply(
        lambda row:
            normalize_text(row[f"{field}_ref"])
            == normalize_text(row[f"{field}_ext"]),
        axis=1
    )

df_validation["all_primary_fields_match"] = (
    matched_mask
    & df_validation[
        PRIMARY_MATCH_COLUMNS
    ].all(axis=1)
)


In [ ]:
# ------------------------------------------------------------
# 10. Record-outcome classification
# ------------------------------------------------------------

def classify_record(row):
    if row["_merge"] == "left_only":
        return "missing"

    if row["_merge"] == "right_only":
        return "unsupported"

    if bool(row["all_primary_fields_match"]):
        return "fully_correct"

    return "discrepant"

df_validation["record_status"] = df_validation.apply(classify_record, axis=1)

missing_records = df_validation[
    df_validation["record_status"] == "missing"
].copy()

unsupported_records = df_validation[
    df_validation["record_status"] == "unsupported"
].copy()

discrepant_records = df_validation[
    df_validation["record_status"] == "discrepant"
].copy()

fully_correct_records_df = df_validation[
    df_validation["record_status"] == "fully_correct"
].copy()

duplicate_extraction_records["record_status"] = "unsupported_duplicate"

print("Missing:", len(missing_records))
print("Unsupported unmatched:", len(unsupported_records))
print("Unsupported duplicate extras:", len(duplicate_extraction_records))
print("Discrepant matched:", len(discrepant_records))
print("Fully correct:", len(fully_correct_records_df))


In [ ]:
# ------------------------------------------------------------
# 11. Validation metrics
# ------------------------------------------------------------

N_REF = int(len(df_ref))
N_EXT = int(len(df_ext))
N_ALIGNED = int((df_validation["_merge"] == "both").sum())
N_MISSING = int(len(missing_records))
N_UNSUPPORTED_UNMATCHED = int(len(unsupported_records))
N_DUPLICATE_EXTRAS = int(len(duplicate_extraction_records))
N_UNSUPPORTED = N_UNSUPPORTED_UNMATCHED + N_DUPLICATE_EXTRAS
N_DISCREPANT = int(len(discrepant_records))
N_CORRECT = int(len(fully_correct_records_df))

completeness = N_ALIGNED / N_REF if N_REF else 0.0

record_precision = N_CORRECT / N_EXT if N_EXT else 0.0
record_recall = N_CORRECT / N_REF if N_REF else 0.0
record_f1 = (
    2 * record_precision * record_recall / (record_precision + record_recall)
    if (record_precision + record_recall) > 0 else 0.0
)

unsupported_rate = (
    N_UNSUPPORTED / N_EXT
    if N_EXT else 0.0
)
discrepancy_rate = N_DISCREPANT / N_ALIGNED if N_ALIGNED else 0.0

matched_validation = df_validation[df_validation["_merge"] == "both"].copy()

field_accuracy_matched = {}
for field in PRIMARY_CORRECTNESS_FIELDS:
    col = f"{field}_match"
    field_accuracy_matched[field] = (
        float(matched_validation[col].mean())
        if len(matched_validation) else 0.0
    )

correct_primary_field_instances = int(
    matched_validation[
        PRIMARY_MATCH_COLUMNS
    ].sum().sum()
)

evaluated_primary_field_instances = int(
    N_ALIGNED
    * len(PRIMARY_CORRECTNESS_FIELDS)
)

field_accuracy = (
    correct_primary_field_instances
    / evaluated_primary_field_instances
    if evaluated_primary_field_instances
    else 0.0
)

summary = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "reference_records": N_REF,
    "extracted_records": N_EXT,
    "aligned_records": N_ALIGNED,
    "fully_correct_records": N_CORRECT,
    "discrepant_records": N_DISCREPANT,
    "missing_records": N_MISSING,
    "unsupported_records": N_UNSUPPORTED,
    "unsupported_unmatched_records": N_UNSUPPORTED_UNMATCHED,
    "unsupported_duplicate_records": N_DUPLICATE_EXTRAS,
    "field_accuracy": round(field_accuracy, 4),
    "completeness": round(completeness, 4),
    "record_precision_exact": round(record_precision, 4),
    "record_recall_exact": round(record_recall, 4),
    "record_f1_exact": round(record_f1, 4),
    "unsupported_rate": round(unsupported_rate, 4),
    "discrepancy_rate_among_aligned": round(discrepancy_rate, 4),
    "field_accuracy_among_aligned": {
        k: round(v, 4) for k, v in field_accuracy_matched.items()
    },
    "structural_validity": structural_validity,
    "schema_diagnostics": schema_diagnostics,
    "matching_key_fields": MATCH_KEY_FIELDS,
    "normalisation_note":
        "Deterministic normalisation was applied only to comparison copies; "
        "the preserved Branch A extraction was not modified.",
    "input_provenance": input_provenance
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
# ------------------------------------------------------------
# 12. Field-level error summary
# ------------------------------------------------------------

field_error_summary = []

for field in PRIMARY_CORRECTNESS_FIELDS:
    col = f"{field}_match"

    correct_aligned = int(matched_validation[col].sum())
    incorrect_aligned = int(len(matched_validation) - correct_aligned)

    field_error_summary.append({
        "field": field,
        "aligned_records_evaluated": int(len(matched_validation)),
        "correct_values_among_aligned": correct_aligned,
        "incorrect_values_among_aligned": incorrect_aligned,
        "accuracy_among_aligned":
            round(correct_aligned / len(matched_validation), 4)
            if len(matched_validation) else 0.0
    })

field_error_summary_df = pd.DataFrame(field_error_summary)
field_error_summary_df


In [ ]:
# ------------------------------------------------------------
# 13. Validation integrity checks
# ------------------------------------------------------------

assert N_ALIGNED + N_MISSING == N_REF

assert (
    N_ALIGNED
    + N_UNSUPPORTED_UNMATCHED
    + N_DUPLICATE_EXTRAS
    == N_EXT
)

assert N_CORRECT + N_DISCREPANT == N_ALIGNED

for metric_name, metric_value in {
    "completeness": completeness,
    "record_precision": record_precision,
    "record_recall": record_recall,
    "record_f1": record_f1,
    "unsupported_rate": unsupported_rate,
    "discrepancy_rate": discrepancy_rate,
    "field_accuracy": field_accuracy
}.items():
    assert 0.0 <= metric_value <= 1.0, f"Invalid {metric_name}: {metric_value}"

print("Validation integrity checks passed.")

In [ ]:
# ------------------------------------------------------------
# 14. Export validation artefacts
# ------------------------------------------------------------

df_validation.to_csv(
    OUTPUT_DIR / "D1_branch_A_validation_detailed.csv",
    index=False
)

missing_records.to_csv(
    OUTPUT_DIR / "D1_branch_A_missing_records.csv",
    index=False
)

unsupported_records.to_csv(
    OUTPUT_DIR / "D1_branch_A_unsupported_unmatched_records.csv",
    index=False
)

duplicate_extraction_records.to_csv(
    OUTPUT_DIR / "D1_branch_A_unsupported_duplicate_records.csv",
    index=False
)

discrepant_records.to_csv(
    OUTPUT_DIR / "D1_branch_A_discrepant_records.csv",
    index=False
)

fully_correct_records_df.to_csv(
    OUTPUT_DIR / "D1_branch_A_fully_correct_records.csv",
    index=False
)

field_error_summary_df.to_csv(
    OUTPUT_DIR / "D1_branch_A_field_error_summary.csv",
    index=False
)

with open(
    OUTPUT_DIR / "D1_branch_A_validation_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Validation artefacts saved.")


In [ ]:
# ------------------------------------------------------------
# 15. Download validation artefacts
# ------------------------------------------------------------

for output_file in sorted(OUTPUT_DIR.iterdir()):
    if output_file.is_file():
        files.download(output_file)